<div align="center">

*Intelligence can emerge from nothing more than:*  
*try things, see what happens, do more of what works.*

— Jeremy K. Chen

</div>

# Chapter 1 — Just Watch It Learn

> *An Inspirational Introduction to Artificial Intelligence*

We are not going to start with definitions. We are not going to build up slowly from mathematics. Instead, we are going to do something that would have seemed like science fiction not long ago: watch a computer teach itself to play a perfect game of Tic-Tac-Toe — from scratch, with no human telling it the rules of strategy.

Run the code below. Watch what happens. Then we will understand why.

> 💡 **How to read this chapter:** Don't worry if you don't understand every line yet. Focus on the *shape* of what's happening. The explanation follows the code.

## The Complete Program

Here is the entire agent — about 80 lines of pure Python and NumPy. Run it and watch the output.

In [1]:
from collections import defaultdict
import random

# ── The Game ──────────────────────────────────────────

def empty_board():
    """A board is a tuple of 9 values: 0=empty, 1=X, -1=O"""
    return (0,) * 9

def winner(board):
    """Return 1 if X wins, -1 if O wins, 0 if game continues."""
    lines = [(0,1,2), (3,4,5), (6,7,8),  # rows
             (0,3,6), (1,4,7), (2,5,8),  # cols
             (0,4,8), (2,4,6)]           # diagonals
    for a, b, c in lines:
        if board[a] == board[b] == board[c] != 0:
            return board[a]
    return 0

def available(board):
    return [i for i, v in enumerate(board) if v == 0]

def make_move(board, pos, player):
    b = list(board); b[pos] = player; return tuple(b)

# ── The Agent ─────────────────────────────────────────

class QLearner:
    def __init__(self, player,
                 alpha=0.5,    # learning rate
                 gamma=0.9,    # discount factor
                 epsilon=0.3): # exploration rate
        self.player  = player
        self.alpha   = alpha
        self.gamma   = gamma
        self.epsilon = epsilon
        self.Q = defaultdict(lambda: 0.0)  # Q-table

    def choose(self, board):
        moves = available(board)
        # Explore: pick a random move
        if random.random() < self.epsilon:
            return random.choice(moves)
        # Exploit: pick the best known move
        return max(moves, key=lambda m: self.Q[(board, m)])

    def update(self, board, move, reward, next_board):
        future = max([self.Q[(next_board, m)]
                      for m in available(next_board)],
                     default=0.0)
        key = (board, move)
        self.Q[key] += self.alpha * (
            reward + self.gamma * future - self.Q[key]
        )

# ── Training ──────────────────────────────────────────

def train(episodes=50_000):
    X = QLearner(player=1)
    O = QLearner(player=-1)

    for ep in range(episodes):
        board = empty_board()
        agents = {1: X, -1: O}
        prev = {1: None, -1: None}  # (board, move) each player last saw
        turn = 1

        while True:
            agent = agents[turn]
            move  = agent.choose(board)
            prev[turn] = (board, move)
            board = make_move(board, move, turn)
            w = winner(board)

            if w != 0:  # someone won
                agents[w].update(*prev[w],   reward=+1.0, next_board=board)
                agents[-w].update(*prev[-w], reward=-1.0, next_board=board)
                break
            elif not available(board):  # draw
                for a in agents.values():
                    a.update(*prev[a.player], reward=0.2, next_board=board)
                break
            else:  # game continues — small penalty for slow play
                agent.update(*prev[turn], reward=-0.01, next_board=board)

            turn = -turn  # switch players

    # After training, stop exploring
    X.epsilon = 0
    O.epsilon = 0
    return X, O

X_agent, O_agent = train(episodes=50_000)
print("Training complete. Agents have played 50,000 games.")

Training complete. Agents have played 50,000 games.


## Watch a Game

In [2]:
def show_board(board):
    symbols = {1: 'X', -1: 'O', 0: '·'}
    row = lambda i: ' | '.join(symbols[board[i*3+j]] for j in range(3))
    print(f'{row(0)}\n──────\n{row(1)}\n──────\n{row(2)}\n')

def play_game(agent_x, agent_o, verbose=True):
    board = empty_board()
    agents = {1: agent_x, -1: agent_o}
    turn = 1
    while True:
        move = agents[turn].choose(board)
        board = make_move(board, move, turn)
        if verbose:
            print(f"{'X' if turn==1 else 'O'} plays position {move}:")
            show_board(board)
        w = winner(board)
        if w != 0: return w
        if not available(board): return 0
        turn = -turn

result = play_game(X_agent, O_agent)
outcome = {1: "X wins!", -1: "O wins!", 0: "Draw — both agents played perfectly."}
print(outcome[result])

X plays position 0:
X | · | ·
──────
· | · | ·
──────
· | · | ·

O plays position 1:
X | O | ·
──────
· | · | ·
──────
· | · | ·

X plays position 2:
X | O | X
──────
· | · | ·
──────
· | · | ·

O plays position 3:
X | O | X
──────
O | · | ·
──────
· | · | ·

X plays position 4:
X | O | X
──────
O | X | ·
──────
· | · | ·

O plays position 7:
X | O | X
──────
O | X | ·
──────
· | O | ·

X plays position 6:
X | O | X
──────
O | X | ·
──────
X | O | ·

X wins!


## Stress Test: Trained Agent vs. Random Opponent

In [3]:
# Play 1,000 games against a random opponent. How often does X win or draw?
random_agent = QLearner(player=-1, epsilon=1.0)  # always random

wins, draws, losses = 0, 0, 0
for _ in range(1_000):
    r = play_game(X_agent, random_agent, verbose=False)
    if   r ==  1: wins   += 1
    elif r == -1: losses += 1
    else:         draws  += 1

print(f"vs. random opponent (1,000 games):")
print(f"  Wins:   {wins/10:.1f}%")
print(f"  Draws:  {draws/10:.1f}%")
print(f"  Losses: {losses/10:.1f}%")

vs. random opponent (1,000 games):
  Wins:   90.9%
  Draws:  6.9%
  Losses: 2.2%


The trained agent rarely loses. Against a random opponent it wins around 90-92% of the time and draws around 6-7% of the time — not losing in about **98% of games**. Against another trained agent it most likely draws, which is the *mathematically correct* outcome for two perfect players.

## What Just Happened?

The agent maintains a table of guesses — one for every **(board state, move)** pair it has ever seen — rating how good that move was in that situation. This is called the **Q-table**, and the algorithm is called **Q-learning**.

Every reinforcement learning system has three elements:

| Element | In our program |
|---|---|
| **Agent** — the decision-maker | `QLearner` |
| **Environment** — the world it acts in | The Tic-Tac-Toe board |
| **Reward** — feedback signal | +1 win, −1 loss, 0.2 draw |

The agent sees the current board (its **state**), picks a square (its **action**), and eventually receives a **reward**. Its goal is to maximize total reward over the game.

### The Update Rule

The single most important line is:

```python
self.Q[key] += self.alpha * (reward + self.gamma * future - self.Q[key])
```

In plain English: *"Nudge my current estimate toward a better estimate — which is the reward I just received, plus a discounted guess of how good the future looks from here."*

The three parameters:

- **`alpha` (learning rate, 0.5)** — How much to trust new information vs. old.
- **`gamma` (discount factor, 0.9)** — How much to value future rewards. A reward 2 steps away is worth 0.9² = 81% of the same reward now.
- **`epsilon` (exploration rate, 0.3)** — Probability of choosing a random move. Without exploration the agent never discovers better strategies.

> ⚠️ **The Exploration-Exploitation Tradeoff:** 70% of the time, be smart (exploit). 30% of the time, be curious (explore). After training, `epsilon=0` — the agent stops exploring and plays its best game.

## The Big Idea

You've built a *learning system*, not just a Tic-Tac-Toe agent. The same architecture — agent, environment, reward, Q-table, update rule — powers DeepMind's AlphaGo, robots learning to walk, and recommendation systems.

---
**The Central Insight of Reinforcement Learning**

> You do not need to tell an agent *how* to be intelligent. You only need to tell it *what* success looks like. Given enough experience and a clear reward signal, intelligence emerges on its own.

---

In Chapter 2 we'll map out the three major branches of machine learning — reinforcement, supervised, and unsupervised — and see how each one uses a different notion of what *learning* means.

## Exercises

The best way to understand a system is to break it.

1. **Change `epsilon` to `0.0` from the start.** Train for 50,000 games. What happens to the win rate? Why?
2. **Train for only 500 episodes.** How does win rate change? Plot win rate vs. training episodes.
3. **Change the draw reward from `0.2` to `-0.5`.** Does the agent take more risks?
4. ⭐ **Add a human player mode** — input a position (0–8) and play against the trained agent. Can you beat it?

In [ ]:
# ⭐ Exercise 4 starter: Human vs. Agent
def play_vs_human(agent):
    board = empty_board()
    print("You are O. Agent is X. Positions are numbered 0-8:")
    print("0 | 1 | 2\n──────\n3 | 4 | 5\n──────\n6 | 7 | 8\n")
    turn = 1
    while True:
        if turn == 1:
            move = agent.choose(board)
            print(f"Agent plays {move}:")
        else:
            show_board(board)
            move = int(input("Your move (0-8): "))
            while move not in available(board):
                move = int(input("Invalid. Try again: "))
        board = make_move(board, move, turn)
        show_board(board)
        w = winner(board)
        if w == 1:  print("Agent wins!"); return
        if w == -1: print("You win!");    return
        if not available(board): print("Draw!"); return
        turn = -turn

# Uncomment to play:
# play_vs_human(X_agent)

---

## Further Reading & Acknowledgments

### For Graduate-Level Readers

This chapter introduced Q-learning through a concrete, runnable example. Readers who want the full mathematical treatment — including Markov Decision Processes, convergence proofs, temporal-difference learning, and policy gradient methods — should go directly to the canonical graduate text:

> Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press. Freely available at [incompleteideas.net/book/the-book.html](http://incompleteideas.net/book/the-book.html)

Sutton & Barto's Chapter 1 also uses Tic-Tac-Toe as its opening example — through a value-function approach rather than Q-learning — and remains the definitive starting point for anyone pursuing RL research.

### Additional References

- Watkins, C. J. C. H., & Dayan, P. (1992). Q-learning. *Machine Learning, 8*(3–4), 279–292. *(The original paper introducing Q-learning.)*

- Mnih, V., et al. (2015). Human-level control through deep reinforcement learning. *Nature, 518*, 529–533. *(DeepMind's DQN paper — Q-learning scaled to Atari games.)*

- Silver, D., et al. (2016). Mastering the game of Go with deep neural networks and tree search. *Nature, 529*, 484–489. *(AlphaGo — the system referenced in this chapter.)*

---

*© 2026 Jeremy K. Chen. All rights reserved.*  
*Published at [kasperwave.ai](https://kasperwave.ai) and
[kasperwave-ai on GitHub](https://github.com/kasperwave-ai*